# 02 — PauliLCU resource estimation

Map each active-space Hamiltonian to qubits via Jordan–Wigner and estimate
the cost of qubitized phase estimation under the PauliLCU encoding.

The qubit Hamiltonian after Jordan–Wigner is

$$\hat{H}_q = \sum_{\ell=1}^{L} \alpha_\ell\, \hat{P}_\ell, \qquad \hat{P}_\ell \in \{I,X,Y,Z\}^{\otimes N}.$$

**Identity-term correction.** The 1-norm $\lambda = \sum_\ell |\alpha_\ell|$
should exclude the identity-Pauli coefficient: that term is a constant energy
offset and does not need to be block-encoded. Including it inflates $\lambda$
by ~1–2 orders of magnitude and makes the QPE bit count and T-cost meaningless.

This notebook reports only the corrected estimates.

In [1]:
import json
import os
import sys
import numpy as np
from openfermion import InteractionOperator, jordan_wigner, count_qubits

sys.path.insert(0, ".")
from config import load_config, ensure_dirs
cfg = load_config()
ensure_dirs(cfg)

## Load active-space integrals from notebook 01

In [2]:
npz_path = os.path.join(cfg["paths"]["data_dir"], "integrals.npz")
npz = np.load(npz_path)
labels = [a["label"] for a in cfg["active_spaces"]]
all_data = {}
for label in labels:
    all_data[label] = {
        "nelec": int(npz[f"{label}_nelec"]),
        "norb": int(npz[f"{label}_norb"]),
        "e_core": float(npz[f"{label}_e_core"]),
        "h1e_eff": npz[f"{label}_h1e_eff"],
        "h2e_phys": npz[f"{label}_h2e_phys"],
    }

## Resource estimation loop

QPE iteration count for additive precision $\varepsilon = 1.6$ mHa:

$$N_{\text{QPE}} = \lceil \pi \lambda / (2\varepsilon) \rceil$$

T-gate cost per walk operator scales linearly with the number of Pauli terms in
the SELECT multiplexer plus a logarithmic PREPARE overhead from QROM.

In [3]:
epsilon = cfg["qpe"]["epsilon_ha"]
rot_eps = cfg["qpe"]["rotation_synthesis_eps"]
results = {}

for label in labels:
    d = all_data[label]
    nelec, norb = d["nelec"], d["norb"]

    iop = InteractionOperator(
        constant=d["e_core"],
        one_body_tensor=d["h1e_eff"],
        two_body_tensor=0.5 * d["h2e_phys"],
    )
    qubit_ham = jordan_wigner(iop)

    # Identity term is a constant offset, not block-encoded
    e_identity = qubit_ham.terms.get((), 0.0)
    lam = sum(abs(v) for k, v in qubit_ham.terms.items() if k != ())
    n_terms = len([k for k in qubit_ham.terms if k != ()])

    # QPE precision bits and rounds
    prec = int(np.ceil(np.log2(lam / epsilon))) + 1
    qpe_rounds = 2 ** prec
    energy_res = lam / (2 ** prec)

    # Walk-operator T-cost (PauliLCU)
    ancilla_prepare = int(np.ceil(np.log2(n_terms)))
    t_per_rotation = int(np.ceil(4 * np.log2(1 / rot_eps)))
    t_prepare = n_terms * t_per_rotation
    t_reflect = ancilla_prepare * t_per_rotation
    t_per_walk = 2 * (t_prepare + t_reflect)
    t_total = qpe_rounds * t_per_walk

    n_qubits = count_qubits(qubit_ham)
    logical_qubits = n_qubits + ancilla_prepare + prec + 1

    results[label] = {
        "active_space": f"({nelec}e, {norb}o)",
        "spin_orbitals": 2 * norb,
        "pauli_terms": int(n_terms),
        "lambda": float(lam),
        "e_identity": float(e_identity),
        "qpe_bits": int(prec),
        "qpe_rounds": int(qpe_rounds),
        "energy_resolution_ha": float(energy_res),
        "t_per_walk": int(t_per_walk),
        "t_total": int(t_total),
        "logical_qubits": int(logical_qubits),
    }

    print(f"({nelec}e,{norb}o) [{label:>7}]: "
          f"L={n_terms:>6,}  lambda={lam:>8.2f}  "
          f"QPE={prec}  T={t_total:>16,}  Q_L={logical_qubits}")

(4e,4o) [  small]: L=    28  lambda=    2.97  QPE=12  T=      35,954,688  Q_L=22
(8e,8o) [ medium]: L=   440  lambda=   15.26  QPE=15  T=   3,913,613,312  Q_L=33
(12e,12o) [ target]: L= 2,340  lambda=   37.43  QPE=16  T=  41,001,418,752  Q_L=41
(16e,16o) [  large]: L= 7,396  lambda=   69.64  QPE=17  T= 258,315,911,168  Q_L=47
(20e,20o) [ xlarge]: L=18,522  lambda=  117.90  QPE=18  T=1,292,590,645,248  Q_L=54
(24e,24o) [xxlarge]: L=38,606  lambda=  184.65  QPE=18  T=2,693,123,801,088  Q_L=59


## Summary table

In [4]:
print(f"{'Space':>10} {'Pauli':>8} {'lambda':>10} {'QPE':>4} "
      f"{'T-gates':>18} {'Logical':>8}")
for label in labels:
    r = results[label]
    print(f"{r['active_space']:>10} {r['pauli_terms']:>8,} "
          f"{r['lambda']:>10.2f} {r['qpe_bits']:>4} "
          f"{r['t_total']:>18,} {r['logical_qubits']:>8}")

     Space    Pauli     lambda  QPE            T-gates  Logical
  (4e, 4o)       28       2.97   12         35,954,688       22
  (8e, 8o)      440      15.26   15      3,913,613,312       33
(12e, 12o)    2,340      37.43   16     41,001,418,752       41
(16e, 16o)    7,396      69.64   17    258,315,911,168       47
(20e, 20o)   18,522     117.90   18  1,292,590,645,248       54
(24e, 24o)   38,606     184.65   18  2,693,123,801,088       59


## Save results

In [5]:
out_path = os.path.join(cfg["paths"]["data_dir"], "pauli_lcu_results.json")
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"saved {out_path}")

saved data/pauli_lcu_results.json
